# from open-meteo API

Fiona 

In [2]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.7/890.7 kB 16.5 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.1/631.1 kB 16.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [openmeteo-requests]iquests]uture]
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests-cache]m [requests-cache]


In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.209272,
	"longitude": 0.129469,
	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", "soil_temperature_54cm", "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm", "soil_moisture_9_to_27cm", "soil_moisture_27_to_81cm"],
	"start_date": "2026-10-20",
	"end_date": "2026-12-01",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {abs(response.Latitude())}°{'S' if response.Latitude() < 0 else 'N'} {abs(response.Longitude())}°{'W' if response.Longitude() < 0 else 'E'}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_80m = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_120m = hourly.Variables(5).ValuesAsNumpy()
hourly_wind_speed_180m = hourly.Variables(6).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_direction_80m = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_direction_120m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_direction_180m = hourly.Variables(10).ValuesAsNumpy()
hourly_soil_temperature_0cm = hourly.Variables(11).ValuesAsNumpy()
hourly_soil_temperature_6cm = hourly.Variables(12).ValuesAsNumpy()
hourly_soil_temperature_18cm = hourly.Variables(13).ValuesAsNumpy()
hourly_soil_temperature_54cm = hourly.Variables(14).ValuesAsNumpy()
hourly_soil_moisture_0_to_1cm = hourly.Variables(15).ValuesAsNumpy()
hourly_soil_moisture_1_to_3cm = hourly.Variables(16).ValuesAsNumpy()
hourly_soil_moisture_3_to_9cm = hourly.Variables(17).ValuesAsNumpy()
hourly_soil_moisture_9_to_27cm = hourly.Variables(18).ValuesAsNumpy()
hourly_soil_moisture_27_to_81cm = hourly.Variables(19).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["rain"] = hourly_rain
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
hourly_data["wind_speed_120m"] = hourly_wind_speed_120m
hourly_data["wind_speed_180m"] = hourly_wind_speed_180m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
hourly_data["wind_direction_120m"] = hourly_wind_direction_120m
hourly_data["wind_direction_180m"] = hourly_wind_direction_180m
hourly_data["soil_temperature_0cm"] = hourly_soil_temperature_0cm
hourly_data["soil_temperature_6cm"] = hourly_soil_temperature_6cm
hourly_data["soil_temperature_18cm"] = hourly_soil_temperature_18cm
hourly_data["soil_temperature_54cm"] = hourly_soil_temperature_54cm
hourly_data["soil_moisture_0_to_1cm"] = hourly_soil_moisture_0_to_1cm
hourly_data["soil_moisture_1_to_3cm"] = hourly_soil_moisture_1_to_3cm
hourly_data["soil_moisture_3_to_9cm"] = hourly_soil_moisture_3_to_9cm
hourly_data["soil_moisture_9_to_27cm"] = hourly_soil_moisture_9_to_27cm
hourly_data["soil_moisture_27_to_81cm"] = hourly_soil_moisture_27_to_81cm

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 52.20757293701172°N 0.1414642333984375°E
Elevation: 5.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                           date  temperature_2m  relative_humidity_2m  rain  \
0    2026-10-20 00:00:00+00:00          12.702             93.000000   0.1   
1    2026-10-20 01:00:00+00:00          12.652             94.000000   0.1   
2    2026-10-20 02:00:00+00:00          12.752             94.000000   0.2   
3    2026-10-20 03:00:00+00:00          12.602             95.000000   0.1   
4    2026-10-20 04:00:00+00:00          12.402             93.000000   0.1   
...                        ...             ...                   ...   ...   
1027 2026-12-01 19:00:00+00:00           5.500             89.433884   0.0   
1028 2026-12-01 20:00:00+00:00           5.100             90.351059   0.0   
1029 2026-12-01 21:00:00+00:00           4.750             91.285957   0.0   
1030 2026-12-01 22:00:00+00:00           4.450             91.912766   0.0   
1031 2026-12-01 23:0

In [2]:
# save data to CSV
hourly_dataframe.to_csv("hourly_weather_data.csv", index=False)